In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ==============================================================================
# STEP 2: DATA LOADING (Wave 4 - 2011)
# ==============================================================================
# ใช้ convert_categoricals=False เพื่อป้องกัน Error จาก Duplicate Labels
file_2011 = 'wave-4-shocksclean.dta'
df_11 = pd.read_stata(file_2011, convert_categoricals=False)

print(f"Loaded Wave 4 (2011) successfully: {len(df_11)} rows")

# ==============================================================================
# STEP 3: DATA CLEANING (Standard Research Logic)
# ==============================================================================
def clean_wave4(df):
    # จัดการ Missing Values ตามมาตรฐาน Stata
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # จัดการคอลัมน์การเงิน (Loss Amount: _x31005a)
    # ปี 2011 อาจมี 'none' หรือข้อมูลประเภท string ปนอยู่
    if '_x31005a' in df.columns:
        df['_x31005a'] = df['_x31005a'].astype(str).str.replace('none', '0', case=False).str.strip()
        df['_x31005a'] = pd.to_numeric(df['_x31005a'], errors='coerce').fillna(0)
    
    return df

df_11 = clean_wave4(df_11)

# ==============================================================================
# STEP 4: COPING STRATEGY HARMONIZATION (เจาะลึก _x31008 - _x31010)
# ==============================================================================
# Mapping รหัสรับมือปี 2011 ให้สอดคล้องกับ Master Template
coping_map_11 = {
    11: 'sold_assets', 12: 'sold_assets', 13: 'sold_assets', 14: 'sold_assets',
    15: 'used_savings', 16: 'used_insurance',
    17: 'borrowed_informal', 18: 'borrowed_informal',
    21: 'borrowed_formal', 22: 'borrowed_formal', 23: 'borrowed_formal',
    28: 'gov_help', 29: 'gov_help', 30: 'relatives_help'
}

# สร้างคอลัมน์ Binary (0/1)
coping_cols = ['sold_assets', 'used_savings', 'used_insurance', 'borrowed_informal', 'borrowed_formal', 'gov_help']
for c in coping_cols:
    df_11[f'coping_{c}'] = 0

# วนลูปตรวจสอบจากลำดับการรับมือทั้ง 3 ช่อง
for col in ['_x31008', '_x31009', '_x31010']:
    if col in df_11.columns:
        for code, name in coping_map_11.items():
            df_11.loc[df_11[col] == code, f'coping_{name}'] = 1

# ==============================================================================
# STEP 5: SHOCK GROUPING (Categorization)
# ==============================================================================
# ใช้รหัสเดียวกับมาตรฐานงานวิจัย (Harmonization)
shock_map_11 = {
    10: 'agricultural', 11: 'agricultural', 63: 'agricultural', 55: 'agricultural',
    1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
    5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
    8: 'social', 70: 'social', 77: 'economics'
}

df_11['shocks_Group'] = df_11['_x31002'].map(shock_map_11).fillna('others')
df_11['survey_year'] = 2011

# ==============================================================================
# STEP 6: EXPORT
# ==============================================================================
output_file = 'shocks_2011_cleaned_final.csv'
df_11.to_csv(output_file, index=False)

# สรุปภาพรวมด้วยกราฟ
sns.countplot(data=df_11, x='shocks_Group', palette='rocket')
plt.title('Shock Distribution (Wave 4 - 2011)')
plt.show()

print(f"✅ Process Completed. File saved as {output_file}")

In [ ]:
import os

def run_master_descriptive(df, year):
    # 1. สร้างโฟลเดอร์เก็บรูป
    output_dir = f"Descriptive_Graph_{year}"
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    sns.set_theme(style="whitegrid")

    # 2. AUTO-DETECT COLUMNS
    # ตรวจหาคอลัมน์เงิน
    c1 = '_x31005a' if '_x31005a' in df.columns else 'v31105a'
    c2 = '_x31005b' if '_x31005b' in df.columns else 'v31105b'
    c3 = '_x31006a' if '_x31006a' in df.columns else 'v31106a'
    
    # ตรวจหาคอลัมน์ Recovery & Consumption
    recovery_col = '_x31012a' if '_x31012a' in df.columns else ('_x31012' if '_x31012' in df.columns else 'v31112a')
    cons_col = '_x31011' if '_x31011' in df.columns else 'v31111'

    # 3. PRE-PROCESSING
    for c in [c1, c2, c3]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace('none', '0'), errors='coerce').fillna(0)
    
    df['impact_3way'] = df[c1] + df[c2] + df[c3]
    df['impact_2way'] = df[c1] + df[c3]

    # Mapping สำหรับ Recovery Standardized (0-3 หรือเดือน -> กลุ่ม)
    def to_standard_recovery(val):
        if pd.isna(val): return np.nan
        # ถ้าเป็นแบบเดือน (Wave 2, 6, 7, 8, 9)
        if recovery_col in ['_x31012a', 'v31112a']:
            if val < 12: return "less than 1 year"
            if val == 12: return "1 year"
            if 12 < val < 90: return "more than 1 year, but recovered"
            if val >= 90: return "not yet recovered"
        # ถ้าเป็นแบบกลุ่ม (Wave 1, 3, 4, 5)
        else:
            m = {0:"less than 1 year", 1:"1 year", 2:"more than 1 year, but recovered", 3:"not yet recovered"}
            return m.get(val, np.nan)
        return np.nan

    df['recovery_std'] = df[recovery_col].apply(to_standard_recovery)
    
    # Consumption Label
    df['cons_label'] = df[cons_col].map({1: "Yes (Reduced)", 2: "No (Did not reduce)"})

    # --- START GENERATING 8 GRAPHS ---
    plots = [
        ('G1_Frequency', lambda: sns.countplot(data=df, x='shocks_Group', palette='viridis')),
        ('G2_Impact_3way', lambda: sns.barplot(data=df, x='shocks_Group', y='impact_3way', estimator=np.mean, palette='magma')),
        ('G3_Loss_2way', lambda: sns.barplot(data=df, x='shocks_Group', y='impact_2way', estimator=np.mean, palette='flare')),
        ('G4_CopingUsage', lambda: df[[c for c in df.columns if c.startswith('coping_')]].sum().sort_values().plot(kind='barh', color='skyblue')),
        ('G5_CopingIntensity', lambda: sns.countplot(data=df, x=df[[c for c in df.columns if c.startswith('coping_')]].sum(axis=1), palette='plasma')),
        ('G6_RecoveryDist', lambda: sns.histplot(data=df[df[recovery_col] < 90], x=recovery_col, bins=20, kde=True) if recovery_col in ['_x31012a', 'v31112a'] else plt.text(0.5,0.5,"N/A for Categorical")),
        ('G7_RecoveryStd', lambda: sns.countplot(data=df, x='recovery_std', order=["less than 1 year", "1 year", "more than 1 year, but recovered", "not yet recovered"], palette='Set2')),
        ('G8_Consumption', lambda: sns.countplot(data=df[df['cons_label'].notnull()], x='cons_label', palette='Set1'))
    ]

    for name, func in plots:
        plt.figure(figsize=(10, 6))
        func()
        plt.title(f"{name.replace('_',' ')} ({year})")
        if 'G1' in name or 'G2' in name or 'G3' in name: plt.xticks(rotation=45)
        plt.savefig(f"{output_dir}/{name}_{year}.png", dpi=300, bbox_inches='tight')
        plt.show()
    
    print(f"✅ All 8 graphs for {year} saved in {output_dir}")

run_master_descriptive(df_11, 2011)